# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from scripts.stoch_sim_model import *

In [3]:
import os
print(os.getcwd())

/mmfs1/home/oukogu/infoimmune


In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-1-'
comment = "acute_all-basic-replicate"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
cell_series_list = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 1248)] if x not in run_list]
print(len(out))
print(' '.join((out)))

0



In [4]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    out = pd.DataFrame(np.hstack((np.array(import_dict["parameters"]), 
                     np.array(import_dict["summary_stats"]))), columns = [i for i in var_names])[keep_vars]
    
    del import_dict

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
# identify Biologically evidenced networks
keep_vars = ['b_I','d_I', 'K_I', 'I_0','S_0', 'N_0', 'd_S', 'K_H'] + Na_reg + NE_reg + EM_reg + EE_reg + ['harm_pI', 'harm_pS', 'max_pE','T_pE_start', 'T_pE_max', 'T_pE_end','E_end']

full_df = pd.concat(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)).groupby(['I_0','d_I','K_I','b_I','K_H','N_0'] + Na_reg + NE_reg + EM_reg + EE_reg, 
                                                                 as_index = False, sort = False).mean()

# # Save datasets
full_df.to_pickle(os.path.join(d, "raw", "stacked_full_data"+runs+"runs"+'-'+comment)+'.pkl')

In [5]:
with pd.option_context('display.max_columns', None):
    display(full_df)

,I_0,d_I,K_I,b_I,K_H,N_0,psi_myc_I,psi_myc_HI,psi_myc_HE,L0_Na,psi_NE_I,psi_NE_HI,psi_NE_HE,L0_NE,psi_EM_I,psi_EM_HI,psi_EM_HE,L0_EM,psi_Edie_I,psi_Edie_HI,psi_Edie_HE,L0_Edie,S_0,d_S,harm_pI,harm_pS,max_pE,T_pE_start,T_pE_max,T_pE_end,E_end
0,1000.0,0.5,10000.0,5.000000e-08,10000.0,100.0,1.5,0.5,0.0,0.0,1.5,0.5,0.0,-1.5,0.0,0.0,0.0,-81.0,-1.5,-0.5,-0.0,-2.5,9999000.0,0.01,1.005000e+03,9.564093e+06,529692.8,1.886,9.104,30.000,1373.4
1,1000.0,0.5,10000.0,5.000000e-08,10000.0,100.0,1.5,0.5,0.0,0.0,1.5,0.5,0.0,-1.5,0.0,0.0,0.0,-81.0,-1.5,-0.5,-0.0,-2.0,9999000.0,0.01,1.005000e+03,8.252002e+06,397703.0,1.780,9.088,27.810,38.4
2,1000.0,0.5,10000.0,5.000000e-08,10000.0,100.0,1.5,0.5,0.0,0.0,1.5,0.5,0.0,-1.5,0.0,0.0,0.0,-81.0,-1.5,-0.5,-0.0,-1.5,9999000.0,0.01,1.005000e+03,6.346527e+06,280311.8,1.942,9.712,22.570,0.0
3,1000.0,0.5,10000.0,5.000000e-08,10000.0,100.0,1.5,0.5,0.0,0.0,1.5,0.5,0.0,-1.5,0.0,0.0,0.0,-81.0,-1.5,-0.5,-0.0,-1.0,9999000.0,0.01,1.004999e+03,4.269023e+06,144268.4,1.842,11.390,22.110,0.0
4,1000.0,0.5,10000.0,5.000000e-08,10000.0,100.0,1.5,0.5,0.0,0.0,1.5,0.5,0.0,-1.5,0.0,0.0,0.0,-81.0,-1.5,-0.5,-0.0,-0.5,9999000.0,0.01,1.005000e+03,2.070547e+06,45758.0,2.372,16.140,26.944,3.2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390971524,1000.0,0.5,10000000.0,2.500000e-07,10000.0,100.0,0.0,0.0,3.0,2.0,0.0,0.0,3.0,-2.0,0.0,0.0,0.0,-81.0,-0.0,-0.0,-3.0,2.5,9999000.0,0.01,9.930311e+06,1.170346e+03,121.0,15.482,4.018,4.848,0.0
390971525,1000.0,0.5,10000000.0,2.500000e-07,10000.0,100.0,0.0,0.0,3.0,2.0,0.0,0.0,3.0,-2.0,0.0,0.0,0.0,-81.0,-0.0,-0.0,-3.0,3.0,9999000.0,0.01,9.930309e+06,1.169151e+03,110.4,10.100,4.578,5.008,0.0
390971526,1000.0,0.5,10000000.0,2.500000e-07,10000.0,100.0,0.0,0.0,3.0,2.0,0.0,0.0,3.0,-1.5,0.0,0.0,0.0,-81.0,-0.0,-0.0,-3.0,-3.0,9999000.0,0.01,5.316792e+06,4.683218e+06,1889230.6,1.376,10.090,30.000,85202.0
390971527,1000.0,0.5,10000000.0,2.500000e-07,10000.0,100.0,0.0,0.0,3.0,2.0,0.0,0.0,3.0,-1.5,0.0,0.0,0.0,-81.0,-0.0,-0.0,-3.0,-2.5,9999000.0,0.01,5.201183e+06,4.798824e+06,1924558.2,1.322,10.312,30.000,13381.8


In [6]:
# Enumerate infection conditions
virs = np.unique(full_df[['I_0','d_I','K_I','b_I','K_H','N_0']].values, axis = 0)

In [7]:
# save data sets
full_infection_scenarios = []
mean_of_infection_scenarios = []
std_of_infection_scenarios = []
no_eff_data = [[] for i in np.arange(len(virs))]
b_S = d_S*S_max

for l, (I_0, d_I, K_I, b_I, K_H, N_0) in enumerate(tqdm(virs)):
    data = full_df.loc[(full_df["d_I"] == d_I)*(full_df["K_I"] == K_I)*(full_df["b_I"] == b_I)*(full_df["K_H"] == K_H)*(full_df["N_0"] == N_0)*(full_df["I_0"] == I_0), 
    keep_vars]

    data.loc[:,"peff_infection"] = data['harm_pI'].to_numpy()/S_max
    data.loc[:,"peff_toxicity"] = data['harm_pS'].to_numpy()/S_max
    data.loc[:,"peff_total_harm"] = data['peff_infection'] + data['peff_toxicity']

    mean_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean())
    std_of_infection_scenarios.append(data.groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).std())
    full_infection_scenarios.append(data)

# stack datasets
pd.concat(mean_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(std_of_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/processed_data'+runs+'runs'+'-'+comment+'.pkl')
pd.concat(full_infection_scenarios).to_pickle('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/processed_full_data'+runs+'runs'+'-'+comment+'.pkl')

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/mean/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(mean_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/std/list_processed_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(std_of_infection_scenarios, f)

with open('/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/summary_stats/list_processed_full_data'+runs+'runs'+'-'+comment+'.pkl', 'wb') as f:
    pickle.dump(full_infection_scenarios, f)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 81/81 [29:12<00:00, 21.63s/it]


In [8]:
# Clear memory
del full_df, mean_of_infection_scenarios, std_of_infection_scenarios, full_infection_scenarios